# WiFi CSI Localization (x, y, z)
**Dataset**: HALOC — 138,879 samples, 128 subcarriers, x/y/z coordinates

**Models**: ResNet CNN (regression) + LightGBM (regression)

**Runtime**: L4 GPU

**Input**: Upload `haloc_compact.parquet` (42 MB) to Google Drive first

In [ ]:
!pip install -q lightgbm pyarrow

import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time, gc, os, json, shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Mount Drive and copy file to local (MUCH faster I/O)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

SAVE_DIR = '/content/drive/MyDrive/fyp'
os.makedirs(SAVE_DIR, exist_ok=True)

# Find file in Drive
DRIVE_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'haloc_compact.parquet' in files:
        DRIVE_PATH = os.path.join(root, 'haloc_compact.parquet')
        break
print(f'Drive: {DRIVE_PATH}')

# Copy to local SSD for fast reads
LOCAL_PATH = '/content/haloc_compact.parquet'
if not os.path.exists(LOCAL_PATH):
    print('Copying to local storage...')
    shutil.copy(DRIVE_PATH, LOCAL_PATH)
print(f'Local: {os.path.getsize(LOCAL_PATH)/1e6:.1f} MB')

In [ ]:
# Load data — 128 subcarriers (2.4 GHz WiFi)
N_SUB = 128
amp_cols = [f'amp_{i}' for i in range(N_SUB)]
phase_cols = [f'phase_{i}' for i in range(N_SUB)]
table = pq.read_table(LOCAL_PATH, columns=amp_cols + phase_cols + ['x', 'y', 'z', 'split'])

# Features
X = np.empty((len(table), N_SUB * 2), dtype=np.float32)
for i, col in enumerate(amp_cols + phase_cols):
    X[:, i] = table.column(col).to_numpy().astype(np.float32)

# Targets: x, y, z
y_x = table.column('x').to_numpy().astype(np.float32)
y_y = table.column('y').to_numpy().astype(np.float32)
y_z = table.column('z').to_numpy().astype(np.float32)
Y = np.column_stack([y_x, y_y, y_z])

# Use HALOC's own split
split = table.column('split').to_pylist()
idx_train = np.array([i for i, s in enumerate(split) if s == 'train'])
idx_val = np.array([i for i, s in enumerate(split) if s == 'val'])
idx_test = np.array([i for i, s in enumerate(split) if s == 'test'])

del table; gc.collect()
print(f'Loaded: {X.shape[0]:,} rows, {X.shape[1]} features')
print(f'Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')
print(f'\nTarget ranges:')
print(f'  x: {y_x.min():.2f} to {y_x.max():.2f} m (range: {y_x.max()-y_x.min():.2f} m)')
print(f'  y: {y_y.min():.2f} to {y_y.max():.2f} m (range: {y_y.max()-y_y.min():.2f} m)')
print(f'  z: {y_z.min():.2f} to {y_z.max():.2f} m (range: {y_z.max()-y_z.min():.2f} m)')

In [ ]:
# Visualize target distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, vals, name in zip(axes, [y_x, y_y, y_z], ['x', 'y', 'z']):
    ax.hist(vals, bins=50, color='steelblue', edgecolor='white')
    ax.set_title(f'{name} distribution')
    ax.set_xlabel(f'{name} (meters)')
plt.tight_layout(); plt.show()

# 2D scatter of x vs y
plt.figure(figsize=(10, 4))
plt.scatter(y_x[::10], y_y[::10], s=1, alpha=0.3)
plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.title('Location distribution (x vs y)'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Feature engineering
amp = X[:, :N_SUB]
phase = X[:, N_SUB:]

feat_eng = np.column_stack([
    amp.mean(1, keepdims=True), amp.std(1, keepdims=True),
    amp.max(1, keepdims=True), amp.min(1, keepdims=True),
    np.median(amp, 1, keepdims=True),
    phase.mean(1, keepdims=True), phase.std(1, keepdims=True),
    # 8 groups of 16 subcarriers
    *[amp[:, i*16:(i+1)*16].mean(1, keepdims=True) for i in range(8)],
    *[amp[:, i*16:(i+1)*16].std(1, keepdims=True) for i in range(8)],
    np.diff(amp, axis=1).mean(1, keepdims=True),
    np.diff(amp, axis=1).std(1, keepdims=True),
    (amp**2).sum(1, keepdims=True),
])
X_full = np.concatenate([X, feat_eng], axis=1)
del amp, phase, feat_eng; gc.collect()
print(f'Features after engineering: {X_full.shape[1]}')

# Normalize features
X_mean = X_full.mean(0)
X_std = X_full.std(0) + 1e-8
X_full = (X_full - X_mean) / X_std

# Normalized raw for CNN
X_norm_raw = X_full[:, :N_SUB*2].copy()
print(f'LightGBM features: {X_full.shape[1]}, CNN features: {X_norm_raw.shape[1]}')

## Model 1: LightGBM Regression

In [ ]:
%%time
# Train separate LightGBM for each coordinate
params_lgb = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 50,
    'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'device': 'gpu', 'gpu_use_dp': False,
    'verbose': -1, 'n_jobs': -1, 'seed': 42,
}

lgb_models = {}
lgb_preds = np.empty((len(idx_test), 3), dtype=np.float32)

for i, coord in enumerate(['x', 'y', 'z']):
    print(f'\nTraining LightGBM for {coord}...')
    ds_train = lgb.Dataset(X_full[idx_train], label=Y[idx_train, i])
    ds_val = lgb.Dataset(X_full[idx_val], label=Y[idx_val, i], reference=ds_train)

    model = lgb.train(
        params_lgb, ds_train, num_boost_round=1000,
        valid_sets=[ds_val], valid_names=['val'],
        callbacks=[lgb.log_evaluation(200), lgb.early_stopping(50)],
    )
    lgb_models[coord] = model
    lgb_preds[:, i] = model.predict(X_full[idx_test])
    print(f'  {coord} best iteration: {model.best_iteration}')

In [ ]:
# Evaluate LightGBM
Y_test = Y[idx_test]

print(f'\n{"="*55}')
print(f'{"Coord":<8} {"MAE (m)":>10} {"RMSE (m)":>10} {"R²":>10}')
print(f'{"="*55}')
lgb_metrics = {}
for i, coord in enumerate(['x', 'y', 'z']):
    mae = mean_absolute_error(Y_test[:, i], lgb_preds[:, i])
    rmse = np.sqrt(mean_squared_error(Y_test[:, i], lgb_preds[:, i]))
    r2 = r2_score(Y_test[:, i], lgb_preds[:, i])
    lgb_metrics[coord] = {'mae': mae, 'rmse': rmse, 'r2': r2}
    print(f'{coord:<8} {mae:>10.4f} {rmse:>10.4f} {r2:>10.4f}')

# Overall 3D distance error
dist_err = np.sqrt(((lgb_preds - Y_test)**2).sum(axis=1))
lgb_metrics['3d_dist'] = {'mae': dist_err.mean(), 'median': np.median(dist_err), 'p90': np.percentile(dist_err, 90)}
print(f'{"="*55}')
print(f'3D distance error: mean={dist_err.mean():.4f}m, median={np.median(dist_err):.4f}m, 90th={np.percentile(dist_err, 90):.4f}m')

## Model 2: ResNet CNN Regression

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + residual)

class CSI_ResNet_Loc(nn.Module):
    def __init__(self, n_sub=128, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.layer1 = nn.Sequential(ResBlock(64), ResBlock(64), nn.MaxPool1d(2))
        self.expand1 = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.layer2 = nn.Sequential(ResBlock(128), ResBlock(128), nn.MaxPool1d(2))
        self.expand2 = nn.Sequential(nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.layer3 = nn.Sequential(ResBlock(256), ResBlock(256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 3),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.expand1(self.layer1(x))
        x = self.expand2(self.layer2(x))
        x = self.pool(self.layer3(x))
        return self.regressor(x)

resnet_model = CSI_ResNet_Loc(n_sub=N_SUB).to(device)
print(f'ResNet params: {sum(p.numel() for p in resnet_model.parameters()):,}')

In [ ]:
# Dataloaders
def make_ds(indices):
    d = X_norm_raw[indices]
    return TensorDataset(
        torch.tensor(np.stack([d[:, :N_SUB], d[:, N_SUB:]], axis=1)),
        torch.tensor(Y[indices])
    )

train_ds = make_ds(idx_train)
val_ds = make_ds(idx_val)
test_ds = make_ds(idx_test)
del X, X_norm_raw; gc.collect()

BS = 2048
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
# Train ResNet — 80 epochs
EPOCHS = 80
criterion = nn.MSELoss()
optimizer = optim.AdamW(resnet_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=len(train_loader))
scaler = torch.amp.GradScaler()

hist = {'train_loss': [], 'val_loss': [], 'val_mae': []}
best_val_mae = float('inf')

for epoch in range(EPOCHS):
    t0 = time.time()
    resnet_model.train()
    tloss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            loss = criterion(resnet_model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tloss += loss.item() * len(xb)
    tloss /= len(train_loader.dataset)

    resnet_model.eval()
    vloss, all_preds, all_labels = 0, [], []
    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = resnet_model(xb)
            vloss += criterion(out, yb).item() * len(xb)
            all_preds.append(out.cpu().numpy())
            all_labels.append(yb.cpu().numpy())
    vloss /= len(val_loader.dataset)
    vp = np.concatenate(all_preds)
    vl = np.concatenate(all_labels)
    vmae = np.abs(vp - vl).mean()

    hist['train_loss'].append(tloss)
    hist['val_loss'].append(vloss)
    hist['val_mae'].append(vmae)

    tag = ''
    if vmae < best_val_mae:
        best_val_mae = vmae
        torch.save(resnet_model.state_dict(), f'{SAVE_DIR}/best_resnet_loc.pt')
        tag = ' *best*'

    if (epoch+1) % 5 == 0 or tag:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | TL: {tloss:.6f} | VL: {vloss:.6f} | VMAE: {vmae:.4f}m | {time.time()-t0:.1f}s{tag}')

print(f'\nBest val MAE: {best_val_mae:.4f}m')

In [ ]:
# ResNet test evaluation
resnet_model.load_state_dict(torch.load(f'{SAVE_DIR}/best_resnet_loc.pt', weights_only=True))
resnet_model.eval()
preds, labels = [], []
with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
    for xb, yb in test_loader:
        preds.append(resnet_model(xb.to(device)).cpu().numpy())
        labels.append(yb.numpy())
resnet_preds = np.concatenate(preds)
resnet_labels = np.concatenate(labels)

print(f'\n{"="*55}')
print(f'{"Coord":<8} {"MAE (m)":>10} {"RMSE (m)":>10} {"R²":>10}')
print(f'{"="*55}')
resnet_metrics = {}
for i, coord in enumerate(['x', 'y', 'z']):
    mae = mean_absolute_error(resnet_labels[:, i], resnet_preds[:, i])
    rmse = np.sqrt(mean_squared_error(resnet_labels[:, i], resnet_preds[:, i]))
    r2 = r2_score(resnet_labels[:, i], resnet_preds[:, i])
    resnet_metrics[coord] = {'mae': mae, 'rmse': rmse, 'r2': r2}
    print(f'{coord:<8} {mae:>10.4f} {rmse:>10.4f} {r2:>10.4f}')

dist_err_resnet = np.sqrt(((resnet_preds - resnet_labels)**2).sum(axis=1))
resnet_metrics['3d_dist'] = {'mae': float(dist_err_resnet.mean()), 'median': float(np.median(dist_err_resnet)), 'p90': float(np.percentile(dist_err_resnet, 90))}
print(f'{"="*55}')
print(f'3D distance error: mean={dist_err_resnet.mean():.4f}m, median={np.median(dist_err_resnet):.4f}m, 90th={np.percentile(dist_err_resnet, 90):.4f}m')

# Save LightGBM models
for coord, model in lgb_models.items():
    model.save_model(f'{SAVE_DIR}/lgb_loc_{coord}.txt')

# Save normalization stats
np.savez(f'{SAVE_DIR}/loc_norm_stats.npz', mean=X_mean, std=X_std)

# Convert history to plain floats
hist_clean = {k: [float(v) for v in vals] for k, vals in hist.items()}

# Save results
results = {
    'task': 'localization',
    'dataset': 'HALOC',
    'n_subcarriers': N_SUB,
    'models': {
        'LightGBM': {
            coord: {k: round(float(v), 6) for k, v in lgb_metrics[coord].items()}
            for coord in ['x', 'y', 'z', '3d_dist']
        },
        'ResNet CNN': {
            coord: {k: round(float(v), 6) for k, v in resnet_metrics[coord].items()}
            for coord in ['x', 'y', 'z', '3d_dist']
        },
    },
    'best_model': 'ResNet CNN' if resnet_metrics['3d_dist']['mae'] < lgb_metrics['3d_dist']['mae'] else 'LightGBM',
    'resnet_history': hist_clean,
}
with open(f'{SAVE_DIR}/loc_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nSaved to: {SAVE_DIR}/')
for fname in sorted(os.listdir(SAVE_DIR)):
    if 'loc' in fname or 'resnet_loc' in fname:
        size = os.path.getsize(os.path.join(SAVE_DIR, fname))
        print(f'  {fname} ({size/1e6:.1f} MB)' if size > 1e6 else f'  {fname} ({size/1e3:.0f} KB)')

In [ ]:
# Compare models
print(f'\n{"="*60}')
print(f'{"":<15} {"LightGBM":>20} {"ResNet CNN":>20}')
print(f'{"="*60}')
for coord in ['x', 'y', 'z']:
    print(f'{coord + " MAE (m)":<15} {lgb_metrics[coord]["mae"]:>20.4f} {resnet_metrics[coord]["mae"]:>20.4f}')
print(f'{"3D dist (m)":<15} {lgb_metrics["3d_dist"]["mae"]:>20.4f} {resnet_metrics["3d_dist"]["mae"]:>20.4f}')
print(f'{"="*60}')

# Scatter: predicted vs actual
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, coord in enumerate(['x', 'y', 'z']):
    # LightGBM
    axes[0, i].scatter(Y_test[:, i], lgb_preds[:, i], s=1, alpha=0.3)
    axes[0, i].plot([Y_test[:, i].min(), Y_test[:, i].max()], [Y_test[:, i].min(), Y_test[:, i].max()], 'r--')
    axes[0, i].set_title(f'LightGBM: {coord} (MAE={lgb_metrics[coord]["mae"]:.3f}m)')
    axes[0, i].set_xlabel('Actual'); axes[0, i].set_ylabel('Predicted')
    # ResNet
    axes[1, i].scatter(resnet_labels[:, i], resnet_preds[:, i], s=1, alpha=0.3)
    axes[1, i].plot([resnet_labels[:, i].min(), resnet_labels[:, i].max()], [resnet_labels[:, i].min(), resnet_labels[:, i].max()], 'r--')
    axes[1, i].set_title(f'ResNet: {coord} (MAE={resnet_metrics[coord]["mae"]:.3f}m)')
    axes[1, i].set_xlabel('Actual'); axes[1, i].set_ylabel('Predicted')
plt.tight_layout(); plt.show()

# 3D error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(dist_err, bins=50, color='steelblue', edgecolor='white', alpha=0.7, label='LightGBM')
axes[0].hist(dist_err_resnet, bins=50, color='green', edgecolor='white', alpha=0.7, label='ResNet')
axes[0].set_xlabel('3D Distance Error (m)'); axes[0].set_ylabel('Count')
axes[0].set_title('Error Distribution'); axes[0].legend()

# Training curves
axes[1].plot(hist['val_mae'], color='green')
axes[1].axhline(best_val_mae, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_mae:.4f}m')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val MAE (m)'); axes[1].legend()
axes[1].set_title('ResNet Val MAE'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Save Models to Google Drive /fyp/

In [ ]:
# Save LightGBM models
for coord, model in lgb_models.items():
    model.save_model(f'{SAVE_DIR}/lgb_loc_{coord}.txt')

# Save normalization stats
np.savez(f'{SAVE_DIR}/loc_norm_stats.npz', mean=X_mean, std=X_std)

# Save results
results = {
    'task': 'localization',
    'dataset': 'HALOC',
    'n_subcarriers': N_SUB,
    'models': {
        'LightGBM': {
            coord: {k: round(float(v), 6) for k, v in lgb_metrics[coord].items()}
            for coord in ['x', 'y', 'z', '3d_dist']
        },
        'ResNet CNN': {
            coord: {k: round(float(v), 6) for k, v in resnet_metrics[coord].items()}
            for coord in ['x', 'y', 'z', '3d_dist']
        },
    },
    'best_model': 'ResNet CNN' if resnet_metrics['3d_dist']['mae'] < lgb_metrics['3d_dist']['mae'] else 'LightGBM',
    'resnet_history': hist,
}
with open(f'{SAVE_DIR}/loc_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nSaved to: {SAVE_DIR}/')
for fname in sorted(os.listdir(SAVE_DIR)):
    if 'loc' in fname or 'resnet_loc' in fname:
        size = os.path.getsize(os.path.join(SAVE_DIR, fname))
        print(f'  {fname} ({size/1e6:.1f} MB)' if size > 1e6 else f'  {fname} ({size/1e3:.0f} KB)')

## Summary — Copy this!

In [ ]:
print('=== COPY EVERYTHING BELOW ===')
print(json.dumps(results, indent=2))